In [1]:
5+5

10

In [2]:

import re
from tokenizers import Regex
from tokenizers.normalizers import NFC, Replace, Sequence


# ============================================================
# TYPE B — lightweight, embeddable normalization
# ============================================================
# We build the normalization as tokenizer components so the *exact same*
# transformation can be (a) embedded in the tokenizer and (b) reused as a
# plain Python function. There is only one definition, so they cannot drift.
#
# Arabic normalization decisions (each is a real trade-off):
#   • remove tashkeel (diacritics) + tatweel  → collapses spelling variants
#   • unify alef  أ إ آ → ا                    → fewer variants, small quality cost
#   • alef maqsura ى → ي                       → common, consistent
#   • we deliberately KEEP ta-marbuta (ة)      → dropping it would stop the model
#                                                from ever generating ة (bad output)

def build_normalizer():

    return Sequence([
        NFC(),
        Replace(Regex( r"[\u064B-\u0652\u0640]"), "" ),
        Replace("أ", "ا"),
        Replace("إ", "ا"),
        Replace("آ", "ا"),
        Replace("ى", "ي"),   
        Replace(Regex(r"[ \t]+"), " "),  # collapse spaces/tabs (keep newlines)
    ])

_NORMALIZER = build_normalizer()

def normalize_text(text):
    """
    Lightweight normalization for a single string.
    """

    return _NORMALIZER.normalize_str(text)

In [10]:
def demo_normalizer():
    print("TYPE B NORMALIZER (embeddable, runs at inference too)")

    samples = [
        "جُمْهُورِيَّةُ مِصْرَ  الْعَرَبِيَّة",      # heavy diacritics
        "أحمد إبراهيم آدم علىّ",                   # alef + ya variants
        "The   United  States",                 # extra spaces
    ]

    for s in samples:
        print(f"  raw       : {s}")
        print(f"  normalized: {normalize_text(s)}")
        print("="*30)

In [11]:
demo_normalizer()

TYPE B NORMALIZER (embeddable, runs at inference too)
  raw       : جُمْهُورِيَّةُ مِصْرَ  الْعَرَبِيَّة
  normalized: جمهورية مصر العربية
  raw       : أحمد إبراهيم آدم علىّ
  normalized: احمد ابراهيم ادم علي
  raw       : The   United  States
  normalized: The United States
